In [ ]:
!pip install gradio python-docx PyMuPDF pandas

# --- Imports ---
import gradio as gr
import docx
import fitz  # PyMuPDF for PDF parsing
import pandas as pd
import re
import os

# --- Helper Functions to Extract Text ---

# Function to extract text from a .docx file
def extract_text_from_docx(path):
    doc = docx.Document(path)
    return ' '.join([para.text.strip() for para in doc.paragraphs if para.text.strip()])

# Function to extract text from a .pdf file
def extract_text_from_pdf(filepath):
    with fitz.open(filepath) as doc:
        return ' '.join([page.get_text() for page in doc])

# Function to clean text by removing extra spaces
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# --- Core Extraction Logic ---

# Function to extract all target fields based on document text
def extract_fields(text):
    fields = {}

    # Court name extraction
    court_match = re.search(r'(Local Court|Regional Court|Higher Regional Court|Landgericht|Amtsgericht)\s+of\s+([\w\s]+)', text)
    fields['Court'] = court_match.group(0) if court_match else "Not Found"

    # File Number extraction
    file_match = re.search(r'File Name:\s*(\w+)', text)
    fields['File Number'] = file_match.group(1) if file_match else "Not Found"

    # Claimant Name and Address extraction
    claimant_match = re.search(r'Lawsuit\s*of\s*(Mr\.|Ms\.|Mrs\.)\s*([^,]+),\s*([^\n]+)', text)
    if claimant_match:
        fields['Name of Claimant'] = claimant_match.group(2).strip()
        fields['Address of Claimant'] = claimant_match.group(3).strip()
    else:
        fields['Name of Claimant'] = "Not Found"
        fields['Address of Claimant'] = "Not Found"

    # Assume single claimant for now
    fields['Number of Claimants'] = "1"
    fields['Type of Claimant (Natural vs Legal Person)'] = "Natural Person"

    # Representation Check
    if 'No legal representative' in text:
        fields['Is the Claimant Represented?'] = "No"
        fields['Name of the Legal Representative'] = "None"
        fields['Address of the Legal Representative'] = "None"
    else:
        rep_match = re.search(r'Legal Representative:\s*(?:Attorney\s*)?(Dr\.)?\s*([^,\n]+),\s*([^\n]+)', text)
        if rep_match:
            fields['Is the Claimant Represented?'] = "Yes"
            fields['Name of the Legal Representative'] = rep_match.group(2).strip()
            fields['Address of the Legal Representative'] = rep_match.group(3).strip()
        else:
            fields['Is the Claimant Represented?'] = "No"
            fields['Name of the Legal Representative'] = "None"
            fields['Address of the Legal Representative'] = "None"

    # Money Claimed extraction
    money_match = re.search(r'EUR\s*([0-9,.]+)', text)
    fields['Money'] = f"EUR {money_match.group(1)}" if money_match else "Not Found"

    # Type of Action (based on keywords)
    lower_text = text.lower()
    if 'subsequent performance' in lower_text:
        fields['Action'] = "Subsequent Performance"
    elif 'withdrawal' in lower_text:
        fields['Action'] = "Withdrawal Confirmation"
    elif 'pain and suffering' in lower_text or 'immaterial damages' in lower_text:
        fields['Action'] = "Compensation for Injury"
    else:
        fields['Action'] = "Other"

    # Check for Removal of Disturbance
    fields['Remove a Disturbance'] = "Yes" if any(word in lower_text for word in ['defect', 'repair', 'remove disturbance']) else "No"

    # Assume no Desist action unless mentioned
    fields['Desist'] = "No"

    # BMW Involvement Check
    fields['BMW Involved?'] = "Yes" if 'BMW' in text else "No"
    fields['BMW w Non-BMW'] = "BMW"
    fields['BMW and Other Parties'] = "Only BMW"

    # Legal Basis Extraction (Sections from BGB or ProdHaftG)
    basis_match = re.findall(r'\u00a7\s*[0-9]+(?:\s*(?:para\.|Abs\.)\s*[0-9]+)?\s*(?:BGB|ProdHaftG)', text)
    fields['Legal Basis for the Claim'] = ', '.join(basis_match) if basis_match else "Not Found"

    # Contractual and Statutory Claims
    if 'purchase contract' in lower_text or 'leasing agreement' in lower_text:
        fields['Contractual Claims'] = "Yes"
    else:
        fields['Contractual Claims'] = "No"

    if 'BGB' in text or 'ProdHaftG' in text:
        fields['Statutory Claims'] = "Yes"
    else:
        fields['Statutory Claims'] = "No"

    # Procedural Claims Check
    fields['Procedural Claims'] = "Yes" if 'costs of the proceedings' in lower_text else "No"

    return fields

# --- Main Upload + Extract Handler ---

# Function that handles uploaded files
def upload_and_extract(file):
    if file is None:
        return pd.DataFrame([{"Field": "Error", "Extracted Information": "No file uploaded."}])

    filepath = file.name if hasattr(file, 'name') else file

    try:
        if filepath.endswith('.txt'):
            with open(filepath, 'r', encoding='utf-8') as f:
                document_text = f.read()
        elif filepath.endswith('.docx'):
            document_text = extract_text_from_docx(filepath)
        elif filepath.endswith('.pdf'):
            document_text = extract_text_from_pdf(filepath)
        else:
            return pd.DataFrame([{"Field": "Error", "Extracted Information": "Unsupported file format."}])

        # Clean the text and extract fields
        document_text = clean_text(document_text)
        fields = extract_fields(document_text)

        # Create a DataFrame for display
        output_df = pd.DataFrame(list(fields.items()), columns=["Field", "Extracted Information"])
        return output_df

    except Exception as e:
        return pd.DataFrame([{"Field": "Error", "Extracted Information": str(e)}])

# --- Gradio Web App Setup ---

with gr.Blocks() as demo:
    gr.Markdown("# 🧾 Legal Claim Extractor - Upload and Get Structured Fields")

    uploader = gr.File(label="Upload Legal Document", file_types=[".pdf", ".docx", ".txt"])
    output_table = gr.DataFrame(label="Extracted Fields")

    upload_btn = gr.Button("Analyze")
    upload_btn.click(upload_and_extract, inputs=uploader, outputs=output_table)

    demo.launch(share=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.2 MB/s eta 0:00:00
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92fc6438864f80be2e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
